# Minimal TSMNet demo notebook for inter-session/-subject source-free (SF) offline and online unsupervised domain adaptation (UDA)

In [1]:
import geoopt
import torch
import sklearn
import spdnets.functionals as fn
from sklearn.datasets import make_classification
from typing import Optional, Tuple
from sklearn.linear_model import LogisticRegressionCV

## Parameters

In [2]:
# network and training configuration

if torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')

# Generate sample data
n_dim = 2
n_components = n_dim * (n_dim+1) // 2
n_informative = 1
n_redundant = 0
n_repeated = 0
n_classes = 2
n_domains = 2
class_sep= 5
n_samples = 250 * n_classes * n_domains
control_frechet_variance = True

## Generative model

In [3]:
# brain signal source generation
z, y_true = make_classification(
    n_samples=n_samples, n_features=n_components, n_informative=n_informative, n_redundant=n_redundant,n_repeated=n_repeated,n_classes=n_classes, n_clusters_per_class=1,random_state=None, class_sep=class_sep, shuffle=False,flip_y=0, shift=0.0, scale=1.0       
)
z = torch.from_numpy(z)
# shuffle components across dimensions
z = z[:, torch.randperm(n_components)]
z = (z - z.mean(dim=0, keepdim=True)) / z.var(dim=0, keepdim=True).sqrt()
y_true = torch.from_numpy(y_true)

# distribute data across classes and domains
z = z.reshape((n_classes, n_samples //n_classes // n_domains, n_domains, n_components))
y_true = y_true.reshape((n_classes, n_samples//n_classes // n_domains, n_domains))
ix_domain = torch.zeros_like(y_true)

# generate the forward model

Q = torch.linalg.qr(torch.randn((n_dim,n_dim), dtype=z.dtype)).Q

domain_fwd = Q @ fn.sym_expm.apply(
    fn.upper2sym(torch.randn((n_domains, n_components), dtype=z.dtype))
)

for uinque_domain in range(n_domains):
    ix_domain[...,uinque_domain] = uinque_domain

# generate source signals
E = fn.sym_expm.apply(fn.upper2sym(z))
# project to channel space
C = domain_fwd @ E @ domain_fwd.mT


# flatten to generate the dataset
E = E.reshape(n_samples, n_dim, n_dim)
C = C.reshape(n_samples, n_dim, n_dim)
y_true = y_true.reshape(n_samples)
ix_domain = ix_domain.reshape(n_samples)


## Manifold alignment 

In [4]:
# domain specific spd batch normalization
def forward_dsspdbn(
    C : torch.Tensor,
    ix_domain : torch.Tensor,
    parameter_t : Optional[torch.nn.Parameter] = None,
    C_bias : Optional[geoopt.ManifoldParameter] = None,
    variance : Optional[geoopt.ManifoldParameter] = None,
    control_variance : bool = False,
    return_stats : bool = False,
) -> torch.Tensor:

    S_out = torch.empty_like(C)
    domain_stats = dict()

    for ix_du in ix_domain.unique():

        ixs = torch.nonzero(ix_domain == ix_du).flatten()

        C_domain = C[ixs]

        # estimate Frechet mean
        C_bar = fn.spd_mean_kracher_flow(C_domain,dim=0,return_dist=False)
        if parameter_t is not None:
            C_t_inv_sqrt = fn.sym_powm.apply(C_bar, parameter_t[ix_du]*(-0.5))
        else:
            C_t_inv_sqrt = fn.sym_invsqrtm.apply(C_bar)

        C_domain = C_t_inv_sqrt @ C_domain @ C_t_inv_sqrt

        if C_bias is not None:
            C_bias_sqrt = fn.sym_sqrtm.apply(C_bias[ix_du])
            C_domain = C_bias_sqrt @ C_domain @ C_bias_sqrt
        
        # estimate variance (at current base point)
        var_domain = torch.norm(
            fn.sym_logm.apply(C_domain), p='fro', dim=(-2,-1), keepdim=True
        ).square().mean(dim=0, keepdim=True).squeeze(-1)
        
        domain_stats[ix_du.item()] = (C_bar, var_domain)

        if control_variance:
            # control frechet variance
            if variance is not None:
                s = variance.clip(min=1e-5).rsqrt()
            else:
                s = var_domain.clip(min=1e-5).rsqrt()
            C_domain = fn.sym_powm.apply(C_domain, s)

        S_out[ixs] = fn.sym_logm.apply(C_domain)

    if return_stats:
        return fn.sym2upper(S_out), domain_stats
    else:
        return fn.sym2upper(S_out)

def forward(
    C : torch.Tensor,
    ix_domain : torch.Tensor,
    weight : torch.Tensor,
    bias : torch.Tensor,
    parameter_t : Optional[torch.nn.Parameter] = None,
    C_bias : Optional[geoopt.ManifoldParameter] = None,
) -> torch.Tensor:
    
    v = forward_dsspdbn(C, ix_domain, parameter_t=parameter_t, C_bias=C_bias)
    
    logit = v @ weight - bias
    return logit

# fit tangent space shared linear classifier
def fit_linear_classifier(x : torch.Tensor, y : torch.Tensor) -> Tuple[torch.Tensor]:

    estimator = LogisticRegressionCV(Cs=[1e-1, 1.,10.,100.])

    estimator.fit(x, y)

    weight = torch.from_numpy(estimator.coef_.T)
    bias = torch.from_numpy(estimator.intercept_)

    n_classes = y.unique().shape[0]

    if n_classes == 2:
        weight = torch.cat((-weight, weight), dim=1)
        bias = torch.stack((-bias, bias), dim=1)

    return weight, bias

## SFUDA method: RCT

In [5]:
# compute domain-sepcific RCT transform

C_bar = torch.empty((n_domains, n_dim, n_dim), dtype=C.dtype)

for ix_du in ix_domain.unique():
    C_bar[ix_du] = fn.spd_mean_kracher_flow(C[ix_domain == ix_du],dim=0,return_dist=False)

C_bar_inv_sqrt = fn.sym_invsqrtm.apply(C_bar)


# apply RCT transform
V = torch.empty_like(C)

for ix_du in ix_domain.unique():
    V[ix_domain == ix_du] = C_bar_inv_sqrt[ix_du] @ C[ix_domain == ix_du] @ C_bar_inv_sqrt[ix_du]


# get the imbalanced dataset
# dict[domain,class] = ratio for classes whose ratio should be changed
imbalance_ratios={
    (1,1) : 0.1,
}

keep_ixs = []
for ix_d in range(n_domains):
    for ix_c in range(n_classes):
        ratio = imbalance_ratios.get((ix_d, ix_c), 1.0)

        ixs = torch.nonzero((ix_domain == ix_d) & (y_true == ix_c) ).flatten()

        n_samples = ixs.shape[0]

        keep_ixs.append( ixs[:int(n_samples * ratio)])

keep_ixs = torch.cat(keep_ixs)
keep_ixs = torch.sort(keep_ixs).values
        
C = C[keep_ixs]
E = E[keep_ixs]
y_true = y_true[keep_ixs]
ix_domain = ix_domain[keep_ixs]


## evaluation

test_domain = n_domains -1  # use last domain for testing

ix_train = torch.nonzero(ix_domain != test_domain).flatten()
ix_test = torch.nonzero(ix_domain == test_domain).flatten()

## decoder
x_train = forward_dsspdbn(C[ix_train], ix_domain[ix_train])
_, domain_stats_train = forward_dsspdbn(
    C[ix_train],
    ix_domain[ix_train],
    control_variance=control_frechet_variance,
    variance=None,
    return_stats=True
)
frechet_variance = torch.cat([stats[1] for stats in domain_stats_train.values()]).mean(dim=0).detach()
print(f"Estimated Frechet variance: {frechet_variance.item()}")

weight, bias = fit_linear_classifier(
    forward_dsspdbn(
        C[ix_train],
        ix_domain[ix_train],
        control_variance=control_frechet_variance,
        variance=frechet_variance,
    ),
    y_true[ix_train],
)

logits = forward(C, ix_domain, weight, bias)
p_hat = torch.nn.functional.softmax(logits, dim=1)
y_hat = torch.argmax(p_hat, dim=1)

for label, ixs in [('train', ix_train), ('RCT', ix_test)]:
    bacc = sklearn.metrics.balanced_accuracy_score(y_true[ixs], y_hat[ixs])
    print(f'{label:5} : bacc={bacc:.2f}')

Estimated Frechet variance: 3.0011693136478432
train : bacc=1.00
RCT   : bacc=0.91


## SFUDA method: SPDIM(bias)

In [6]:
# define parameters
parameter_t = torch.ones((n_domains)).to(C)
C_bias = geoopt.ManifoldTensor(
    torch.diag_embed(torch.ones(n_domains, n_dim))
).to(C)

parameters = []

# uncomment to learn C_bias
C_bias = geoopt.ManifoldParameter(C_bias)
parameters.append(C_bias)

optimizer = geoopt.optim.RiemannianAdam(parameters, lr=5e-3, weight_decay=0.)
best_loss = float('inf')
best_C_bias = C_bias.detach().clone()
for epoch in range(50):
    optimizer.zero_grad()
    v = forward_dsspdbn(C, ix_domain, parameter_t=parameter_t, C_bias=C_bias)
    
    logits = v @ weight - bias
    loss = fn.im_loss(logits[ix_test])
    loss.backward()
    optimizer.step()

    p_hat = torch.nn.functional.softmax(logits, dim=1)
    y_hat = torch.argmax(p_hat, dim=1)
    bacc_test = sklearn.metrics.balanced_accuracy_score(y_true[ix_test], y_hat[ix_test].detach().numpy())
    if epoch % 10 == 0:
        print(f'epoch={epoch:3d} loss={loss.detach().item():4.2f} t={parameter_t[1].detach().item():4.2f} bacc={bacc_test:.02f}')

    if loss.item() < best_loss:
        best_loss = loss.item()
        best_C_bias = C_bias.clone()

v = forward_dsspdbn(C, ix_domain, parameter_t=parameter_t, C_bias=best_C_bias)
logits = v @ weight - bias
p_hat = torch.nn.functional.softmax(logits, dim=1)
y_hat = torch.argmax(p_hat, dim=1)
bacc_test = sklearn.metrics.balanced_accuracy_score(y_true[ix_test], y_hat[ix_test].detach().numpy())
print(f'SPDIM(bias) bacc={bacc_test:.02f}')

epoch=  0 loss=-0.25 t=1.00 bacc=0.91
epoch= 10 loss=-0.26 t=1.00 bacc=0.93
epoch= 20 loss=-0.27 t=1.00 bacc=0.95
epoch= 30 loss=-0.27 t=1.00 bacc=0.97
epoch= 40 loss=-0.28 t=1.00 bacc=0.98
SPDIM(bias) bacc=0.99
